# 30. 전이학습 — 남의 모델을 내 것으로

> **제30장** · **이론편 대응: 12.5절(전이학습), 19.1절(사전학습-미세조정)**
> **예상 소요**: 80분 (학습 약 5분)
> **필요 사양**: **[CPU]** 로 실행 가능
> **추가 설치**: 없음
> **다운로드**: ResNet-18 사전학습 가중치 약 45MB (자동)

---

## 이 장에서 하는 일

지금까지 두 가지를 했다.

- **밑바닥부터 학습** (15장 CNN) — 데이터가 많이 필요했다
- **남의 모델 그대로 사용** (23장~) — 우리 문제에 딱 맞지는 않았다

**그 사이가 전이학습이다.** 남이 학습한 것을 가져와 내 문제에 맞게 고친다.

| 절 | 하는 일 | 이론편 대응 |
|---|---|---|
| 1 | 왜 전이학습인가 | 12.5절 |
| 2 | 사전학습 모델 살펴보기 | 12.5절 |
| 3 | **밑바닥부터 vs 전이학습** ★ | 12.5절 |
| 4 | **특성 추출 vs 미세조정** ★ | 12.5절 |
| 5 | 층을 골라 동결하기 | 12.5절 |
| 6 | 학습률 선택 | 11.1절 |
| 7 | 데이터가 적을 때 | 12.5절 |
| 8 | LLM 시대의 전이학습 | 19.1절 |

**3~4절이 핵심이다.** 같은 데이터·같은 시간으로
**얼마나 차이가 나는지** 직접 측정한다.

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import platform
import time
from pathlib import Path

_c = {"Windows": ["Malgun Gothic"], "Darwin": ["AppleGothic"],
      "Linux": ["NanumGothic", "Noto Sans CJK KR", "Noto Sans CJK JP"]}
_a = {f.name for f in fm.fontManager.ttflist}
for _n in _c.get(platform.system(), []):
    if _n in _a:
        plt.rcParams["font.family"] = _n
        break
plt.rcParams["axes.unicode_minus"] = False

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch {torch.__version__} / 장치: {device}")

import torchvision
print(f"torchvision {torchvision.__version__}")

---

## 1. 왜 전이학습인가 — 이론편 12.5절

**15장에서 CNN을 밑바닥부터 학습시켰다.** 그때는 데이터가 충분했다.
실무에서는 대개 그렇지 않다.

| 상황 | 밑바닥부터 | 전이학습 |
|---|---|---|
| 데이터 100만 장 | 가능 | 불필요할 수도 |
| 데이터 1만 장 | 어려움 | **적합** |
| 데이터 수백 장 | 거의 불가능 | **가능** |
| 학습 시간 | 며칠\~몇 주 | 몇 분\~몇 시간 |

### 왜 되는가

**낮은 층이 배우는 것은 일반적이다.**

```
1층: 가장자리, 색 변화        ← 어떤 이미지든 필요
2층: 모서리, 단순한 무늬
3층: 질감, 부분 형태
...
마지막 층: "이것은 고양이다"   ← 특정 문제에만 해당
```

**앞부분은 그대로 쓰고 뒷부분만 바꾸면 된다.**

In [ ]:
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Subset
from pathlib import Path

root = Path.cwd()
if root.name.startswith("part"):
    root = root.parent

# ImageNet 사전학습 모델에 맞춘 전처리
# - 흑백 → 3채널 (사전학습 모델이 컬러를 기대)
# - 크기 조정
# - ImageNet 통계로 정규화
transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=3),
    transforms.Resize(48),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

print("데이터 준비 중... (15장에서 받았다면 즉시)")
train_full = datasets.FashionMNIST(
    root=str(root / "data"), train=True, download=True, transform=transform)
test_full = datasets.FashionMNIST(
    root=str(root / "data"), train=False, download=True, transform=transform)

N_TRAIN, N_TEST = 400, 600
train_set = Subset(train_full, range(N_TRAIN))
test_set = Subset(test_full, range(N_TEST))

train_loader = DataLoader(train_set, batch_size=32, shuffle=True)
test_loader = DataLoader(test_set, batch_size=64)

print()
print("=" * 70)
print("실습 설정")
print("=" * 70)
print(f"  학습 데이터: {N_TRAIN}장  ← 일부러 적게")
print(f"  시험 데이터: {N_TEST}장")
print(f"  클래스     : 10개")
print()
print("전처리 주의점")
print("  1) 흑백 → 3채널: 사전학습 모델은 컬러 이미지로 학습되었다")
print("  2) ImageNet 정규화: 학습 때 쓴 것과 같은 통계를 써야 한다")
print()
print("  이 두 가지를 빠뜨리면 전이학습 효과가 크게 줄어든다.")

sample_x, sample_y = train_set[0]
print()
print(f"입력 텐서 모양: {tuple(sample_x.shape)}   (채널, 높이, 너비)")

---

## 2. 사전학습 모델 살펴보기 — 이론편 12.5절

**ResNet-18**을 쓴다. ImageNet(1000개 클래스, 120만 장)으로 학습된 모델이다.

23장에서 GPT-2 내부를 열어 봤듯, 여기서도 구조를 확인한다.

In [ ]:
import torch
import torch.nn as nn
from torchvision import models
import time

print("=" * 78)
print("ResNet-18 불러오기")
print("=" * 78)
print("사전학습 가중치 약 45MB 를 내려받습니다 (처음 한 번)")
print()

t0 = time.time()
pretrained = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
print(f"로드 완료: {time.time()-t0:.1f}초")
print()

n_params = sum(p.numel() for p in pretrained.parameters())
print(f"파라미터: {n_params:,}개 ({n_params/1e6:.1f}M)")
print()

print("구조")
for name, module in pretrained.named_children():
    if name.startswith("layer"):
        n_blocks = len(list(module.children()))
        print(f"  {name:<12}{type(module).__name__:<16}블록 {n_blocks}개")
    else:
        print(f"  {name:<12}{type(module).__name__}")
print()

print(f"마지막 층: {pretrained.fc}")
print(f"  → 출력이 1000개다 (ImageNet 클래스 수)")
print(f"  → 우리 문제는 10개이므로 이 층을 바꿔야 한다")

In [ ]:
import torch
import numpy as np

print("=" * 78)
print("층별 파라미터 분포")
print("=" * 78)

groups = {}
for name, param in pretrained.named_parameters():
    key = name.split(".")[0]
    groups[key] = groups.get(key, 0) + param.numel()

total = sum(groups.values())
print(f"{'층':<16}{'파라미터':<16}{'비율':<12}{'막대'}")
print("-" * 78)
for name, count in groups.items():
    bar = "█" * int(count / total * 50)
    print(f"{name:<16}{count:<16,}{count/total*100:>6.1f}%     {bar}")
print("-" * 78)
print()
print("뒤로 갈수록 파라미터가 많다 (layer4 가 가장 큼).")
print("  채널 수가 늘어나기 때문이다 — 15장에서 다룬 CNN 구조")
print()
print("[전이학습에서의 의미]")
print("  앞쪽 층은 파라미터가 적고 일반적인 특징을 담는다 → 그대로 쓴다")
print("  뒤쪽 층은 파라미터가 많고 특정 문제에 맞춰져 있다 → 조정한다")

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt

print("=" * 78)
print("사전학습 모델이 무엇을 보는가")
print("=" * 78)

# 첫 번째 합성곱 층의 필터를 시각화
first_conv = pretrained.conv1
weights = first_conv.weight.data.clone()

print(f"conv1 필터: {tuple(weights.shape)}")
print(f"  → 필터 {weights.shape[0]}개, 각 {weights.shape[2]}x{weights.shape[3]}, 채널 {weights.shape[1]}")
print()

# 정규화해 시각화
w_min, w_max = weights.min(), weights.max()
weights_norm = (weights - w_min) / (w_max - w_min)

fig, axes = plt.subplots(4, 8, figsize=(12, 6))
for i, ax in enumerate(axes.flat):
    if i < weights_norm.shape[0]:
        img = weights_norm[i].permute(1, 2, 0).numpy()
        ax.imshow(img)
    ax.axis("off")

fig.suptitle("ResNet-18 의 첫 층 필터 32개 (ImageNet 으로 학습됨)", fontsize=12)
plt.tight_layout()
plt.show()

print("무엇이 보이는가")
print("  - 방향이 있는 가장자리 (수직·수평·대각선)")
print("  - 색 대비 패턴")
print("  - 점 모양")
print()
print("**이런 것들은 어떤 이미지에나 필요하다.**")
print("  옷 사진이든 동물 사진이든 가장자리는 똑같이 중요하다.")
print("  → 이것이 전이학습이 작동하는 이유다 (이론편 12.5절)")

---

## 3. 밑바닥부터 vs 전이학습 ★ — 이론편 12.5절

**같은 데이터, 같은 에폭으로 비교한다.**

| 방식 | 시작점 | 학습 대상 |
|---|---|---|
| 밑바닥부터 | 무작위 초기화 | 전체 |
| 전이학습 | ImageNet 가중치 | 전체 (미세조정) |

In [ ]:
import torch
import torch.nn as nn
from torchvision import models
import time


def evaluate(model, loader):
    """정확도 계산"""
    model.eval()
    correct = total = 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            correct += (model(x).argmax(1) == y).sum().item()
            total += len(y)
    return correct / total


def train_model(model, epochs=5, lr=1e-3, params=None, verbose=True):
    """학습하며 에폭마다 정확도를 기록한다"""
    model = model.to(device)
    optimizer = torch.optim.Adam(params or model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()

    history = []
    t0 = time.time()

    for epoch in range(epochs):
        model.train()
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()
            criterion(model(x), y).backward()
            optimizer.step()

        acc = evaluate(model, test_loader)
        history.append(acc)
        if verbose:
            print(f"    에폭 {epoch+1}: 시험 정확도 {acc:.4f}  "
                  f"({time.time()-t0:.0f}초)")

    return history, time.time() - t0


EPOCHS = 3

print("=" * 78)
print("[1] 밑바닥부터 학습")
print("=" * 78)
torch.manual_seed(0)
scratch = models.resnet18(weights=None)          # 가중치 없이
scratch.fc = nn.Linear(512, 10)
hist_scratch, time_scratch = train_model(scratch, EPOCHS, lr=1e-3)

print()
print("=" * 78)
print("[2] 전이학습 — 사전학습 가중치로 시작")
print("=" * 78)
torch.manual_seed(0)
finetune = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
finetune.fc = nn.Linear(512, 10)                 # 마지막 층만 교체
hist_finetune, time_finetune = train_model(finetune, EPOCHS, lr=1e-4)

print()
print("=" * 78)
print("비교")
print("=" * 78)
print(f"{'방식':<20}{'1에폭 후':<14}{'최종':<14}{'소요 시간'}")
print("-" * 78)
print(f"{'밑바닥부터':<20}{hist_scratch[0]:<14.4f}{hist_scratch[-1]:<14.4f}{time_scratch:.0f}초")
print(f"{'전이학습':<20}{hist_finetune[0]:<14.4f}{hist_finetune[-1]:<14.4f}{time_finetune:.0f}초")
print("-" * 78)
print(f"차이: {hist_finetune[-1] - hist_scratch[-1]:+.4f}")
print()
print(f"[주목] 전이학습은 **1에폭 만에** {hist_finetune[0]:.4f} 를 넘는다")
print(f"  밑바닥부터는 5에폭을 써도 {hist_scratch[-1]:.4f} 다")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))

# --- 왼쪽: 학습 곡선 ---
ax = axes[0]
epochs_x = range(1, EPOCHS + 1)
ax.plot(epochs_x, hist_scratch, marker="o", linewidth=2.5,
        color="#DC2626", label="밑바닥부터")
ax.plot(epochs_x, hist_finetune, marker="s", linewidth=2.5,
        color="#0D9488", label="전이학습")
ax.set_xlabel("에폭")
ax.set_ylabel("시험 정확도")
ax.set_xticks(list(epochs_x))
ax.set_title("같은 데이터, 같은 에폭")
ax.legend(fontsize=9)
ax.grid(alpha=0.3)

# --- 오른쪽: 최종 성능 ---
ax = axes[1]
labels = ["밑바닥부터", "전이학습"]
finals = [hist_scratch[-1], hist_finetune[-1]]
firsts = [hist_scratch[0], hist_finetune[0]]

x = np.arange(2)
w = 0.36
ax.bar(x - w/2, firsts, w, label="1에폭 후", color="#94A3B8")
ax.bar(x + w/2, finals, w, label="5에폭 후", color="#1E40AF")
for i, (f1, f5) in enumerate(zip(firsts, finals)):
    ax.text(i - w/2, f1 + 0.015, f"{f1:.3f}", ha="center", fontsize=9)
    ax.text(i + w/2, f5 + 0.015, f"{f5:.3f}", ha="center", fontsize=9)
ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.set_ylabel("시험 정확도")
ax.set_ylim(0, 1.0)
ax.set_title("출발점의 차이")
ax.legend(fontsize=9)
ax.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.show()

print("=" * 78)
print("왜 이런 차이가 나나")
print("=" * 78)
print()
print("  밑바닥부터: 가장자리 검출부터 배워야 한다")
print("             수백 장으로는 그것조차 제대로 배우기 어렵다")
print()
print("  전이학습  : 가장자리·질감은 이미 안다")
print("             '이 특징들의 조합이 어떤 옷인가'만 배우면 된다")
print()
print("2절에서 본 첫 층 필터를 떠올려 보자.")
print("  그런 필터를 무작위에서 만들어 내려면 많은 데이터가 필요하다.")

---

## 4. 특성 추출 vs 미세조정 ★ — 이론편 12.5절

전이학습에도 두 가지 방식이 있다.

| 방식 | 학습 대상 | 특징 |
|---|---|---|
| **특성 추출** (feature extraction) | **마지막 층만** | 빠름, 메모리 적음 |
| **미세조정** (fine-tuning) | 전체 또는 일부 | 느림, 성능 높음 |

**특성 추출은 사전학습 모델을 "고정된 특징 추출기"로 쓴다.**
앞부분을 얼려(freeze) 그래디언트를 계산하지 않는다.

In [ ]:
import torch
import torch.nn as nn
from torchvision import models
import time

print("=" * 78)
print("[3] 특성 추출 — 마지막 층만 학습")
print("=" * 78)

torch.manual_seed(0)
feature_ext = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

# 전부 얼린다
for param in feature_ext.parameters():
    param.requires_grad = False

# 마지막 층만 새로 만든다 (새 층은 requires_grad=True 가 기본)
feature_ext.fc = nn.Linear(512, 10)

n_trainable = sum(p.numel() for p in feature_ext.parameters() if p.requires_grad)
n_total = sum(p.numel() for p in feature_ext.parameters())

print(f"  전체 파라미터: {n_total:,}")
print(f"  학습 대상    : {n_trainable:,} ({n_trainable/n_total*100:.3f}%)")
print()

hist_feature, time_feature = train_model(
    feature_ext, EPOCHS, lr=1e-3,
    params=feature_ext.fc.parameters())

print()
print("=" * 78)
print("세 방식 비교")
print("=" * 78)
print(f"{'방식':<20}{'학습 파라미터':<20}{'최종 정확도':<16}{'소요 시간'}")
print("-" * 78)
print(f"{'밑바닥부터':<20}{n_total:<20,}{hist_scratch[-1]:<16.4f}{time_scratch:.0f}초")
print(f"{'전이학습 (전체)':<20}{n_total:<20,}{hist_finetune[-1]:<16.4f}{time_finetune:.0f}초")
print(f"{'특성 추출 (동결)':<20}{n_trainable:<20,}{hist_feature[-1]:<16.4f}{time_feature:.0f}초")
print("-" * 78)
print()
print(f"특성 추출은 파라미터의 {n_trainable/n_total*100:.2f}% 만 학습하고")
print(f"시간도 {time_finetune/time_feature:.1f}배 빠르다.")
print()
print("[26장 LoRA 와 비교해 보자]")
print("  LoRA 도 '전체를 얼리고 일부만 학습'하는 발상이다.")
print("  차이: 특성 추출은 마지막 층만 / LoRA 는 모든 층에 작은 어댑터를 붙임")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))

methods = ["밑바닥부터", "전이학습\n(전체)", "특성 추출\n(동결)"]
finals = [hist_scratch[-1], hist_finetune[-1], hist_feature[-1]]
times = [time_scratch, time_finetune, time_feature]
trainables = [n_total, n_total, n_trainable]
colors_m = ["#DC2626", "#0D9488", "#1E40AF"]

# --- 학습 곡선 ---
ax = axes[0]
for hist, label, color in [(hist_scratch, "밑바닥부터", "#DC2626"),
                           (hist_finetune, "전이학습", "#0D9488"),
                           (hist_feature, "특성 추출", "#1E40AF")]:
    ax.plot(range(1, EPOCHS+1), hist, marker="o", linewidth=2.2,
            color=color, label=label)
ax.set_xlabel("에폭")
ax.set_ylabel("시험 정확도")
ax.set_xticks(range(1, EPOCHS+1))
ax.set_title("학습 곡선")
ax.legend(fontsize=8)
ax.grid(alpha=0.3)

# --- 최종 정확도 ---
ax = axes[1]
bars = ax.bar(range(3), finals, color=colors_m)
for b, v in zip(bars, finals):
    ax.text(b.get_x()+b.get_width()/2, v+0.015, f"{v:.3f}",
            ha="center", fontsize=10)
ax.set_xticks(range(3))
ax.set_xticklabels(methods, fontsize=8)
ax.set_ylabel("최종 정확도")
ax.set_ylim(0, 1.0)
ax.set_title("성능")
ax.grid(axis="y", alpha=0.3)

# --- 학습 파라미터 (로그) ---
ax = axes[2]
bars = ax.bar(range(3), trainables, color=colors_m)
for b, v in zip(bars, trainables):
    ax.text(b.get_x()+b.get_width()/2, v*1.3, f"{v/1e6:.2f}M" if v > 1e5 else f"{v:,}",
            ha="center", fontsize=9)
ax.set_yscale("log")
ax.set_xticks(range(3))
ax.set_xticklabels(methods, fontsize=8)
ax.set_ylabel("학습 파라미터 (로그)")
ax.set_title("학습 대상 크기")
ax.grid(axis="y", alpha=0.3, which="both")

plt.tight_layout()
plt.show()

print("=" * 78)
print("어느 것을 쓸까")
print("=" * 78)
print(f"{'상황':<30}{'권장':<20}{'이유'}")
print("-" * 78)
print(f"{'데이터가 아주 적음 (수백)':<30}{'특성 추출':<20}{'과대적합 방지'}")
print(f"{'데이터가 적당함 (수천)':<30}{'미세조정':<20}{'성능이 더 좋다'}")
print(f"{'원본과 도메인이 비슷':<30}{'특성 추출':<20}{'특징이 잘 맞음'}")
print(f"{'원본과 도메인이 다름':<30}{'미세조정':<20}{'특징 조정 필요'}")
print(f"{'계산 자원이 부족':<30}{'특성 추출':<20}{'빠르고 가벼움'}")
print("-" * 78)

---

## 5. 층을 골라 동결하기 — 이론편 12.5절

**전부 얼리거나 전부 푸는 것 사이**의 선택지가 있다.

```
conv1, layer1  ← 얼림 (일반적인 특징)
layer2, layer3 ← 얼림
layer4         ← 학습 (문제 특화)
fc             ← 학습 (새로 만든 층)
```

**뒤쪽일수록 특정 문제에 맞춰져 있으므로** 뒤쪽부터 푸는 것이 관행이다.

In [ ]:
import torch
import torch.nn as nn
from torchvision import models
import time


def make_partial_model(unfreeze_from=None, seed=0):
    """지정한 층부터 학습 대상으로 만든다

    unfreeze_from: None(전부 동결) / "layer4" / "layer3" / "all"
    """
    torch.manual_seed(seed)
    model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

    if unfreeze_from != "all":
        for param in model.parameters():
            param.requires_grad = False

        if unfreeze_from is not None:
            # 지정한 층부터 뒤쪽을 푼다
            layer_order = ["conv1", "bn1", "layer1", "layer2",
                           "layer3", "layer4"]
            start = layer_order.index(unfreeze_from)
            for name in layer_order[start:]:
                for param in getattr(model, name).parameters():
                    param.requires_grad = True

    model.fc = nn.Linear(512, 10)      # 새 층은 항상 학습
    return model


print("=" * 78)
print("동결 범위에 따른 비교")
print("=" * 78)
print("(각 설정마다 학습하므로 2~3분 걸립니다)")
print()

freeze_configs = [
    ("전부 동결", None, 1e-3),
    ("layer4 부터", "layer4", 3e-4),
    ("전부 학습", "all", 1e-4),
]

freeze_results = []
for name, unfreeze, lr in freeze_configs:
    model = make_partial_model(unfreeze)
    trainable = [p for p in model.parameters() if p.requires_grad]
    n_tr = sum(p.numel() for p in trainable)

    hist, elapsed = train_model(model, EPOCHS, lr=lr,
                                params=trainable, verbose=False)
    freeze_results.append({
        "name": name, "n_trainable": n_tr,
        "acc": hist[-1], "time": elapsed, "history": hist,
    })
    print(f"  {name:<16} 학습 {n_tr:>10,}개  "
          f"정확도 {hist[-1]:.4f}  {elapsed:.0f}초")

print()
print("=" * 78)
print(f"{'설정':<16}{'학습 파라미터':<18}{'비율':<12}{'정확도':<12}{'시간'}")
print("-" * 78)
for r in freeze_results:
    print(f"{r['name']:<16}{r['n_trainable']:<18,}"
          f"{r['n_trainable']/n_total*100:>6.2f}%     "
          f"{r['acc']:<12.4f}{r['time']:.0f}초")
print("-" * 78)
print()
best = max(freeze_results, key=lambda r: r["acc"])
print(f"최고 성능: {best['name']} ({best['acc']:.4f})")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))

names_f = [r["name"] for r in freeze_results]
accs_f = [r["acc"] for r in freeze_results]
params_f = [r["n_trainable"] for r in freeze_results]
times_f = [r["time"] for r in freeze_results]

# --- 왼쪽: 학습 파라미터 vs 정확도 ---
ax = axes[0]
ax.plot(params_f, accs_f, marker="o", markersize=10, linewidth=2,
        color="#1E40AF")
for r in freeze_results:
    ax.annotate(r["name"], (r["n_trainable"], r["acc"]),
                textcoords="offset points", xytext=(8, 6), fontsize=8)
ax.set_xscale("log")
ax.set_xlabel("학습 파라미터 (로그)")
ax.set_ylabel("시험 정확도")
ax.set_title("얼마나 풀어야 할까")
ax.grid(alpha=0.3, which="both")

# --- 오른쪽: 학습 곡선 ---
ax = axes[1]
colors_f = ["#94A3B8", "#0D9488", "#EA580C", "#DC2626"]
for r, color in zip(freeze_results, colors_f):
    ax.plot(range(1, EPOCHS+1), r["history"], marker="o",
            linewidth=2, color=color, label=r["name"])
ax.set_xlabel("에폭")
ax.set_ylabel("시험 정확도")
ax.set_xticks(range(1, EPOCHS+1))
ax.set_title("동결 범위별 학습 곡선")
ax.legend(fontsize=8)
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print("[관찰]")
print("  더 많이 풀수록 항상 좋은 것은 아니다.")
print("  데이터가 적으면 많이 풀었을 때 과대적합할 수 있다 (18장 참조).")
print()
print("[실무의 접근]")
print("  1) 먼저 전부 동결하고 마지막 층만 학습 — 빠르게 기준 확보")
print("  2) 성능이 부족하면 뒤에서부터 한 층씩 풀어 본다")
print("  3) 매번 측정한다 (32장 4절의 방식)")

---

## 6. 학습률 선택 — 이론편 11.1절

**전이학습에서는 작은 학습률을 쓴다.** 왜일까.

사전학습 가중치는 **이미 좋은 위치**에 있다.
큰 학습률로 움직이면 그 좋은 위치를 망친다(catastrophic forgetting).

17장에서 다룬 학습률 선택이 여기서 특히 중요하다.

In [ ]:
import torch
import torch.nn as nn
from torchvision import models
import time

print("=" * 78)
print("학습률에 따른 전이학습 결과")
print("=" * 78)
print("(3가지 학습률로 시험합니다)")
print()

lr_configs = [1e-3, 1e-4, 1e-5]
lr_results = []

for lr in lr_configs:
    torch.manual_seed(0)
    model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
    model.fc = nn.Linear(512, 10)
    hist, elapsed = train_model(model, 2, lr=lr, verbose=False)
    lr_results.append({"lr": lr, "history": hist, "final": hist[-1]})
    print(f"  lr={lr:<8} 1에폭 {hist[0]:.4f}  최종 {hist[-1]:.4f}")

print()
print("=" * 78)
print(f"{'학습률':<14}{'1에폭':<14}{'최종':<14}{'평가'}")
print("-" * 78)
for r in lr_results:
    if r["final"] > 0.8:
        note = "좋음"
    elif r["final"] > 0.7:
        note = "보통"
    else:
        note = "너무 크거나 작음"
    print(f"{r['lr']:<14}{r['history'][0]:<14.4f}{r['final']:<14.4f}{note}")
print("-" * 78)
print()
print("[전이학습의 학습률 관행]")
print(f"{'상황':<30}{'권장 학습률'}")
print("-" * 78)
print(f"{'밑바닥부터 (15장)':<30}1e-3 ~ 1e-2")
print(f"{'전이학습 — 전체 파인튜닝':<30}1e-5 ~ 1e-4")
print(f"{'전이학습 — 마지막 층만':<30}1e-3 ~ 1e-2  (새 층이므로)")
print(f"{'LLM 파인튜닝 (28장)':<30}1e-5 ~ 5e-5")
print("-" * 78)
print()
print("새로 만든 층은 무작위 초기화이므로 큰 학습률이 필요하고,")
print("사전학습된 층은 작은 학습률이 필요하다.")

In [ ]:
import torch
import torch.nn as nn
from torchvision import models
import matplotlib.pyplot as plt
import numpy as np

print("=" * 78)
print("층별로 다른 학습률 (discriminative learning rate)")
print("=" * 78)
print()
print("앞쪽 층일수록 작은 학습률을 준다.")
print()

torch.manual_seed(0)
model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
model.fc = nn.Linear(512, 10)

# 층별로 다른 학습률을 지정한다
param_groups = [
    {"params": model.conv1.parameters(), "lr": 1e-6},
    {"params": model.bn1.parameters(), "lr": 1e-6},
    {"params": model.layer1.parameters(), "lr": 1e-6},
    {"params": model.layer2.parameters(), "lr": 5e-6},
    {"params": model.layer3.parameters(), "lr": 1e-5},
    {"params": model.layer4.parameters(), "lr": 5e-5},
    {"params": model.fc.parameters(), "lr": 1e-3},      # 새 층은 크게
]

print(f"{'층':<16}{'학습률':<16}{'이유'}")
print("-" * 78)
print(f"{'conv1, bn1':<16}{'1e-6':<16}가장 일반적인 특징 — 거의 안 바꿈")
print(f"{'layer1':<16}{'1e-6':<16}일반적")
print(f"{'layer2':<16}{'5e-6':<16}조금 조정")
print(f"{'layer3':<16}{'1e-5':<16}더 조정")
print(f"{'layer4':<16}{'5e-5':<16}문제 특화 — 많이 조정")
print(f"{'fc (새 층)':<16}{'1e-3':<16}무작위 초기화 — 크게 학습")
print("-" * 78)
print()

model = model.to(device)
optimizer = torch.optim.Adam(param_groups)
criterion = nn.CrossEntropyLoss()

import time
t0 = time.time()
hist_discr = []
for epoch in range(2):
    model.train()
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        criterion(model(x), y).backward()
        optimizer.step()
    hist_discr.append(evaluate(model, test_loader))

print(f"층별 학습률 결과: {hist_discr[-1]:.4f}  ({time.time()-t0:.0f}초)")
print(f"단일 학습률(1e-4): {hist_finetune[-1]:.4f}")
print()
print("[이 기법이 유용한 경우]")
print("  원본 도메인과 목표 도메인이 꽤 다를 때")
print("  데이터가 어중간하게 있을 때 (수천 장)")
print()
print("  다만 조정할 것이 늘어나므로, 먼저 단일 학습률로 시도하는 편이 낫다.")

---

## 7. 데이터가 적을 때 — 이론편 12.5절

**전이학습의 진가는 데이터가 적을 때 드러난다.**

데이터 양을 바꿔 가며 두 방식을 비교한다.

In [ ]:
import torch
import torch.nn as nn
from torchvision import models
from torch.utils.data import DataLoader, Subset
import time

print("=" * 78)
print("데이터 양에 따른 차이")
print("=" * 78)
print("(여러 크기로 학습하므로 3~4분 걸립니다)")
print()

data_sizes = [100, 250, 400]
size_results = {"scratch": [], "transfer": []}

for n in data_sizes:
    subset = Subset(train_full, range(n))
    loader = DataLoader(subset, batch_size=32, shuffle=True)

    # 임시로 train_loader 를 교체
    global train_loader
    original_loader = train_loader
    train_loader = loader

    # 밑바닥부터
    torch.manual_seed(0)
    m1 = models.resnet18(weights=None)
    m1.fc = nn.Linear(512, 10)
    h1, _ = train_model(m1, 2, lr=1e-3, verbose=False)

    # 전이학습
    torch.manual_seed(0)
    m2 = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
    m2.fc = nn.Linear(512, 10)
    h2, _ = train_model(m2, 2, lr=1e-4, verbose=False)

    train_loader = original_loader

    size_results["scratch"].append(h1[-1])
    size_results["transfer"].append(h2[-1])
    print(f"  {n:>5}장: 밑바닥 {h1[-1]:.4f}  /  전이 {h2[-1]:.4f}  "
          f"(차이 {h2[-1]-h1[-1]:+.4f})")

print()
print("=" * 78)
print(f"{'데이터 수':<14}{'밑바닥부터':<16}{'전이학습':<16}{'차이'}")
print("-" * 78)
for n, s, t in zip(data_sizes, size_results["scratch"], size_results["transfer"]):
    print(f"{n:<14}{s:<16.4f}{t:<16.4f}{t-s:+.4f}")
print("-" * 78)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))

# --- 왼쪽: 데이터 양에 따른 성능 ---
ax = axes[0]
ax.plot(data_sizes, size_results["scratch"], marker="o", linewidth=2.5,
        color="#DC2626", label="밑바닥부터")
ax.plot(data_sizes, size_results["transfer"], marker="s", linewidth=2.5,
        color="#0D9488", label="전이학습")
ax.set_xlabel("학습 데이터 수")
ax.set_ylabel("시험 정확도")
ax.set_title("데이터가 적을수록 차이가 크다")
ax.legend(fontsize=9)
ax.grid(alpha=0.3)

# --- 오른쪽: 격차 ---
ax = axes[1]
gaps = [t - s for s, t in zip(size_results["scratch"],
                              size_results["transfer"])]
bars = ax.bar(range(len(data_sizes)), gaps, color="#1E40AF")
for b, g in zip(bars, gaps):
    ax.text(b.get_x()+b.get_width()/2, g + 0.005, f"{g:+.3f}",
            ha="center", fontsize=9)
ax.axhline(0, color="black", linewidth=0.8)
ax.set_xticks(range(len(data_sizes)))
ax.set_xticklabels([str(n) for n in data_sizes])
ax.set_xlabel("학습 데이터 수")
ax.set_ylabel("정확도 차이 (전이 − 밑바닥)")
ax.set_title("전이학습의 이득")
ax.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.show()

print("=" * 78)
print("[해석]")
print("  데이터가 적을수록 전이학습의 이득이 크다.")
print("  데이터가 아주 많아지면 밑바닥부터도 따라잡을 수 있다.")
print()
print("  실무에서 데이터가 수십만 장 이상인 경우는 드물다.")
print("  → 대부분의 경우 전이학습이 유리하다")
print()
print("[함께 쓰면 좋은 것]")
print("  18장의 데이터 증강 — 적은 데이터를 더 효과적으로 쓴다")
print("  18장의 조기 종료 — 과대적합을 막는다")

---

## 8. LLM 시대의 전이학습 — 이론편 19.1절

**이 장에서 한 것이 사실 LLM 파인튜닝과 같은 발상이다.**

| | 이미지 (이 장) | LLM (28~30장) |
|---|---|---|
| 사전학습 | ImageNet 120만 장 | 인터넷 텍스트 수조 토큰 |
| 사전학습 과제 | 1000개 클래스 분류 | 다음 토큰 예측 |
| 전이 방식 | 마지막 층 교체 | 그대로 + 지시 학습 |
| 특성 추출 | 앞부분 동결 | **LoRA** (26장) |
| 미세조정 | 전체 학습 | 전체 파인튜닝 |

In [ ]:
print("=" * 78)
print("전이학습의 발전 — 이미지에서 LLM 까지")
print("=" * 78)
print()
print(f"{'시기':<16}{'방식':<30}{'특징'}")
print("-" * 78)
stages = [
    ("~2012", "밑바닥부터 학습", "데이터·자원이 많이 필요"),
    ("2012~", "ImageNet 사전학습 + 미세조정", "이 장에서 한 것"),
    ("2018~", "BERT/GPT 사전학습 + 미세조정", "이론편 19.1절"),
    ("2020~", "프롬프트만으로 (few-shot)", "학습 없이 지시로"),
    ("2021~", "PEFT (LoRA 등)", "26장 — 일부만 학습"),
    ("2022~", "지시 학습 + 정렬", "28·30장"),
]
for a, b, c in stages:
    print(f"{a:<16}{b:<30}{c}")
print("-" * 78)
print()
print("[공통된 발상]")
print("  '많은 데이터로 일반적인 것을 배운 뒤, 적은 데이터로 특화한다'")
print()
print("  이 발상 자체는 이미지에서 먼저 확립되었고,")
print("  LLM 이 그것을 언어로 확장한 것이다 (이론편 19.1절).")
print()
print("=" * 78)
print("이 장과 다음 장들의 대응")
print("=" * 78)
print()
print(f"{'이 장':<28}{'LLM 에서는':<28}{'장'}")
print("-" * 78)
print(f"{'마지막 층 교체':<28}{'분류 헤드 추가':<28}{'23장'}")
print(f"{'특성 추출 (동결)':<28}{'LoRA — 어댑터만 학습':<28}{'26장'}")
print(f"{'전체 파인튜닝':<28}{'전체 파인튜닝':<28}{'28장'}")
print(f"{'작은 학습률':<28}{'5e-5 수준':<28}{'28장'}")
print(f"{'catastrophic forgetting':<28}{'참조 모델 (DPO)':<28}{'30장'}")
print("-" * 78)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

print("=" * 78)
print("자원 관점 비교 — 26장 LoRA 와 나란히")
print("=" * 78)
print()

# 이 장에서 측정한 값
print(f"{'방식':<24}{'학습 파라미터':<20}{'비율'}")
print("-" * 78)
print(f"{'전체 파인튜닝 (ResNet)':<24}{n_total:<20,}{100.0:>6.2f}%")
print(f"{'특성 추출 (ResNet)':<24}{n_trainable:<20,}"
      f"{n_trainable/n_total*100:>6.3f}%")
print()
# 26장의 LoRA 값
d_model, r = 4096, 8
lora_ratio = 2 * d_model * r / (d_model * d_model) * 100
print(f"{'LoRA r=8 (26장)':<24}{'(이론편 22.4절)':<20}{lora_ratio:>6.2f}%")
print("-" * 78)
print()
print("세 방법 모두 '전체를 학습하지 않는다'는 점이 같다.")
print()
print(f"{'':<24}{'특성 추출':<24}{'LoRA'}")
print("-" * 78)
print(f"{'학습 대상':<24}{'마지막 층만':<24}{'모든 층의 작은 어댑터'}")
print(f"{'표현력':<24}{'제한적':<24}{'더 유연'}")
print(f"{'구현':<24}{'requires_grad=False':<24}{'별도 행렬 추가'}")
print(f"{'저장':<24}{'마지막 층':<24}{'어댑터만 (수백 KB)'}")
print("-" * 78)
print()
print("특성 추출은 '마지막에서만' 조정할 수 있다.")
print("LoRA 는 '모든 층에서 조금씩' 조정할 수 있어 더 유연하다.")
print("  → 26장에서 이 차이를 다뤘다")

---

## 9. 정리

### 측정 결과

| 방식 | 학습 파라미터 | 정확도 | 시간 |
|---|---|---|---|
| 밑바닥부터 | 전체 | 낮음 | 김 |
| **전이학습 (전체)** | 전체 | **가장 높음** | 김 |
| 특성 추출 (동결) | **0.05%** | 중간 | **가장 빠름** |

**데이터가 적을수록 전이학습의 이득이 크다.**

### 기억할 것

| 항목 | 요점 |
|---|---|
| 전처리 | 사전학습 때와 **같은 정규화**를 써야 |
| 흑백 이미지 | 3채널로 변환 필요 |
| 마지막 층 | 클래스 수에 맞게 교체 |
| 동결 | `requires_grad = False` |
| 학습률 | 사전학습 층은 **작게** (1e-4~1e-5) |
| 새 층 | 무작위 초기화 → 크게 (1e-3) |
| 층별 학습률 | 앞쪽일수록 작게 |
| 동결 순서 | **뒤에서부터** 풀어 나감 |

### 다음 장들과 이어지는 지점

이 장의 발상이 그대로 이어진다.

| 개념 | 다음 장 |
|---|---|
| 전체 파인튜닝 | 28장 SFT |
| 특성 추출 (일부만 학습) | 26장 LoRA |
| 작은 학습률 | 28장 |
| 원본에서 벗어나지 않기 | 30장 DPO의 참조 모델 |

**"많은 데이터로 일반적인 것을 배운 뒤 적은 데이터로 특화한다"** —
이 발상이 오늘날 AI의 기본 구조다.

### 다음 장

**31. SFT — 지도 미세조정** — 같은 발상을 언어 모델에 적용한다.
이 장의 '마지막 층 교체' 대신 **'지시를 따르는 법'**을 가르친다.